In [1]:
import pandas as pd
import requests

from pathlib import Path

In [2]:
url = "https://raw.githubusercontent.com/anilbhaila/llm-zoomcamp-finalproject/refs/heads/main/data/Ecommerce_FAQ_Chatbot_dataset.json"


In [3]:
def load_data(*args, **kwargs):
    """
    Extract data from URL. 
    
    """
    
    try:
        response = requests.get(url)
        response.raise_for_status()  # Raise an exception for bad status codes
        
        json_data = response.json()

        faqs = json_data.get("questions")
        # Create a DataFrame
        df = pd.DataFrame(list(faqs))

        return df
    except Exception as e:
        print(f"An error occurred while reading the CSV file: {e}")
        return None

In [4]:
df = load_data()
df.head()

,question,answer
0,How can I create an account?,"To create an account, click on the 'Sign Up' b..."
1,What payment methods do you accept?,"We accept major credit cards, debit cards, and..."
2,How can I track my order?,You can track your order by logging into your ...
3,What is your return policy?,Our return policy allows you to return product...
4,Can I cancel my order?,You can cancel your order if it has not been s...


In [5]:
import re


In [54]:
def transformToAddChunk(data: pd.DataFrame, *args, **kwargs):
    """
    Template code for a transformer block to add Chunk.

    """
    # Specify your transformation logic here

    rowNumber = 0
    documents = []

    for _, row in data.iterrows():
        number = str(rowNumber)
        rowNumber+=1
        question = str(row['question'])
        answer = str(row['answer'])

        sanitized_question = re.sub(r'\W', '_', question[:30]).lower()
        document_id = f"doc_{number}_{sanitized_question}"

        # Format the document string
        chunk = '\n'.join([
            f'question:\n{question}\n',
            f'answer:\n{answer}\n',
        ])

        documents.append({
            'chunk': chunk,
            'data': {
                'number': number,
                'question': question,
                'answer': answer
            },
            'document_id': document_id,
        })

    print(f'Documents: {len(documents)}')

    return documents

In [55]:
chunk_documents = transformToAddChunk(df)

Documents: 79


In [56]:
chunk_documents[0]

{'chunk': "question:\nHow can I create an account?\n\nanswer:\nTo create an account, click on the 'Sign Up' button on the top right corner of our website and follow the instructions to complete the registration process.\n",
 'data': {'number': '0',
  'question': 'How can I create an account?',
  'answer': "To create an account, click on the 'Sign Up' button on the top right corner of our website and follow the instructions to complete the registration process."},
 'document_id': 'doc_0_how_can_i_create_an_account_'}

In [57]:
from typing import Dict, List
import spacy

def transformToAddTokensBySpacyNLP(documents: List[Dict], *args, **kwargs):
    """
    Template code for a transformer block to Lemmatize.
    """
    count = len(documents)
    print('Documents', count)

    nlp = spacy.load('en_core_web_sm')
    
    data = []

    for idx, document in enumerate(documents):
        document_id = document['document_id']
        if idx % 100 == 0:
            print(f'{idx + 1}/{count}')

        # Process the text chunk using spacy
        chunk = document['chunk']
        doc = nlp(chunk)
        tokens = [token.lemma_ for token in doc]

        data.append(
            dict(
                chunk=chunk,
                document_id=document_id,
                tokens=tokens,
                question=document['data']['question'],
                answer=document['data']['answer'],
            )
        )

    print('\nData', len(data))

    return data

In [ ]:
lemmatize_documents = transformToAddTokensBySpacyNLP(chunk_documents)
#lemmatize_documents[0]

Documents 79
1/79

Data 79


In [28]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer("all-MiniLM-L6-v2")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
from typing import Dict, List

import numpy as np
import spacy

def transformToAddEmbeddingBySpecy(documents: List[Dict], *args, **kwargs) ->List[Dict]:
    """
    Template code for a transformer block to create embeddings.
    """
    # Specify your transformation logic here
    count = len(documents)
    print('Documents', count)

    data = []

    for idx, document in enumerate(documents):
        document_id = document['document_id']
        if idx % 100 == 0:
            print(f'{idx + 1}/{count}')
        nlp = spacy.load('en_core_web_sm')
        tokens = document['tokens']
    
        # Combine tokens back into a single string of text used for embedding
        text = ' '.join(tokens)
        doc = nlp(text)
    
        # Average the word vectors in the doc to get a general embedding
        embedding = np.mean([token.vector for token in doc], axis=0).tolist()
    
        data.append(dict(
            chunk=document['chunk'],
            document_id=document['document_id'],
            question=document['question'],
            answer=document['answer'],
            embedding=embedding,
        ))

    return data

In [59]:
embedding_documentsBySpecy = transformToAddEmbeddingBySpecy(lemmatize_documents)
len(embedding_documentsBySpecy[0]["embedding"])

Documents 79
1/79


96

In [76]:
from typing import Dict, List

import numpy as np
import spacy

def transformToAddEmbeddingByST(documents: List[Dict], *args, **kwargs) ->List[Dict]:
    """
    Template code for a transformer block to create embeddings by Sentence Transformer.
    """
    # Specify your transformation logic here
    count = len(documents)
    print('Documents', count)

    data = []

    for idx, document in enumerate(documents):
        embedding = model.encode(document['chunk'])
    
        data.append(dict(
            chunk=document['chunk'],
            document_id=document['document_id'],
            question=document['question'],
            answer=document['answer'],
            embedding=embedding,
        ))

    return data

In [77]:
embedding_documentsByST = transformToAddEmbeddingByST(lemmatize_documents)
len(embedding_documentsByST[0]["embedding"])

Documents 79


384

In [82]:
import json

from typing import Dict, List, Union

import numpy as np
from elasticsearch import Elasticsearch

def export_dataToIndex(documents: List[Dict[str, Union[Dict, List[int], str]]], *args, **kwargs):
    """
    Exports data to some source.
    
    """
    # Specify your data exporting logic here
    connection_string = kwargs.get('connection_string', 'http://localhost:9200')
    index_name = kwargs.get('index_name', 'documents')
    number_of_shards = kwargs.get('number_of_shards', 1)
    number_of_replicas = kwargs.get('number_of_replicas', 0)
    dimensions = kwargs.get('dimensions')

    if dimensions is None and len(documents) > 0:
        document = documents[0]
        dimensions = len(document.get('embedding'))
        print(f"Dimensions:{dimensions}")

    es_client = Elasticsearch(connection_string, request_timeout=60.0)

    print(f'Connecting to Elasticsearch at {connection_string}')

    index_settings = {
            "settings": {
                "number_of_shards": number_of_shards,
                "number_of_replicas": number_of_replicas,
            },
            "mappings": {
                "properties": {
                    "chunk": {"type": "text"},
                    "document_id": {"type": "text"},
                    "question": {"type": "text"},
                    "answer": {"type": "text"},
                    "embedding": {
                        "type": "dense_vector", 
                        "dims": dimensions,
                        "index": True,
                        "similarity": "cosine"
                    },
                }
            }
        }
    
    if es_client.indices.exists(index=index_name):
        es_client.indices.delete(index=index_name)
        print(f'Index {index_name} deleted')

    es_client.indices.create(index=index_name, body=index_settings)
    print('Index created with properties:')
    print(json.dumps(index_settings, indent=2))
    print('Embedding dimensions:', dimensions)

    count = len(documents)
    print(f'Indexing {count} documents to Elasticsearch index {index_name}')
    for idx, document in enumerate(documents):
        if idx % 2 == 0:
            print(f'Indexing.. {idx + 1}/{count}')

        if isinstance(document['embedding'], np.ndarray):
            document['embedding'] = document['embedding'].tolist()

        es_client.index(index=index_name, document=document)

    return [d['embedding'] for d in documents[:1]]

In [85]:
stEmbedding_indexed = export_dataToIndex(embedding_documentsByST,index_name="documents_st")
stEmbedding_indexed

Dimensions:384
Connecting to Elasticsearch at http://localhost:9200
Index documents_st deleted
Index created with properties:
{
  "settings": {
    "number_of_shards": 1,
    "number_of_replicas": 0
  },
  "mappings": {
    "properties": {
      "chunk": {
        "type": "text"
      },
      "document_id": {
        "type": "text"
      },
      "question": {
        "type": "text"
      },
      "answer": {
        "type": "text"
      },
      "embedding": {
        "type": "dense_vector",
        "dims": 384,
        "index": true,
        "similarity": "cosine"
      }
    }
  }
}
Embedding dimensions: 384
Indexing 79 documents to Elasticsearch index documents_st
Indexing.. 1/79
Indexing.. 3/79
Indexing.. 5/79
Indexing.. 7/79
Indexing.. 9/79
Indexing.. 11/79
Indexing.. 13/79
Indexing.. 15/79
Indexing.. 17/79
Indexing.. 19/79
Indexing.. 21/79
Indexing.. 23/79
Indexing.. 25/79
Indexing.. 27/79
Indexing.. 29/79
Indexing.. 31/79
Indexing.. 33/79
Indexing.. 35/79
Indexing.. 37/79
Inde

[[-0.00113416719250381,
  -0.12893570959568024,
  -0.021831050515174866,
  0.025953106582164764,
  -0.044022880494594574,
  0.01838076300919056,
  0.007050255313515663,
  0.029378293082118034,
  0.018643060699105263,
  0.0022506918758153915,
  -0.03759143128991127,
  -0.06683804094791412,
  0.06715783476829529,
  -0.007149779703468084,
  0.03026438318192959,
  -0.030028803274035454,
  -0.10149011760950089,
  -0.026231851428747177,
  0.009549440816044807,
  0.029605424031615257,
  0.05041773244738579,
  -0.0965995043516159,
  -0.050033845007419586,
  0.009134603664278984,
  -0.02239670790731907,
  -0.06525212526321411,
  0.05307159945368767,
  0.06391441076993942,
  0.02493230625987053,
  0.07959099858999252,
  0.11508239805698395,
  -0.07984435558319092,
  0.050062939524650574,
  -0.03444482013583183,
  -0.008694842457771301,
  -0.020360922440886497,
  -0.08820892870426178,
  -0.0004854442668147385,
  -0.036247141659259796,
  0.0001137500221375376,
  -0.07987833023071289,
  -0.07695985

In [ ]:
specyEmbedding_indexed = export_dataToIndex(embedding_documentsBySpecy,index_name="documents_spacy")
specyEmbedding_indexed

Dimensions:96
Connecting to Elasticsearch at http://localhost:9200
Index created with properties:
{
  "settings": {
    "number_of_shards": 1,
    "number_of_replicas": 0
  },
  "mappings": {
    "properties": {
      "chunk": {
        "type": "text"
      },
      "document_id": {
        "type": "text"
      },
      "question": {
        "type": "text"
      },
      "answer": {
        "type": "text"
      },
      "embedding": {
        "type": "dense_vector",
        "dims": 96,
        "index": true,
        "similarity": "cosine"
      }
    }
  }
}
Embedding dimensions: 96
Indexing 79 documents to Elasticsearch index documents_specy
Indexing.. 1/79
Indexing.. 3/79
Indexing.. 5/79
Indexing.. 7/79
Indexing.. 9/79
Indexing.. 11/79
Indexing.. 13/79
Indexing.. 15/79
Indexing.. 17/79
Indexing.. 19/79
Indexing.. 21/79
Indexing.. 23/79
Indexing.. 25/79
Indexing.. 27/79
Indexing.. 29/79
Indexing.. 31/79
Indexing.. 33/79
Indexing.. 35/79
Indexing.. 37/79
Indexing.. 39/79
Indexing.. 41/

[[-0.21343107521533966,
  -0.5658838152885437,
  0.08929453790187836,
  -0.021446753293275833,
  -0.11743324995040894,
  0.04470497742295265,
  0.2153482884168625,
  0.03439966216683388,
  -0.037103597074747086,
  0.1286245882511139,
  -0.0429355725646019,
  0.14869722723960876,
  0.13001644611358643,
  0.21767203509807587,
  0.5346853137016296,
  0.08476872742176056,
  -0.1593695729970932,
  -0.1102224662899971,
  0.2835747301578522,
  0.0744214653968811,
  0.02511248178780079,
  0.4940701723098755,
  0.11377029865980148,
  -0.0924975648522377,
  0.3497902750968933,
  -0.07383574545383453,
  0.2835349142551422,
  0.011244055815041065,
  0.09735128283500671,
  0.2033621072769165,
  -0.009408318437635899,
  0.14112089574337006,
  0.29277804493904114,
  -0.21818038821220398,
  0.22006221115589142,
  -0.036354970186948776,
  0.02638719417154789,
  -0.07717086374759674,
  -0.033911097794771194,
  -0.10444222390651703,
  -0.12649233639240265,
  0.2698240876197815,
  0.2567797601222992,
  -0

In [88]:
es_client = Elasticsearch('http://localhost:9200') 

index_name='documents_st'
try:
    result = es_client.count(index=index_name)
    print(f"ES Checking = Document count in {index_name}: {result['count']}")
except Exception as e:
    print(f"ES Checking = Error: {str(e)}")

ES Checking = Document count in documents_st: 79


In [ ]:
es_client = Elasticsearch('http://localhost:9200') 

index_name='documents_spacy'
try:
    result = es_client.count(index=index_name)
    print(f"ES Checking = Document count in {index_name}: {result['count']}")
except Exception as e:
    print(f"ES Checking = Error: {str(e)}")

ES Checking = Document count in documents_specy: 79


In [89]:
def get_vector(query):
    doc = nlp(query)
    tokens = [token.lemma_ for token in doc]
    text = ' '.join(tokens)
    doc_lemmatized = nlp(text)
    vector = np.mean([token.vector for token in doc_lemmatized], axis=0).tolist()
    return vector

In [90]:
query = "How can i create my account?"
query

'How can i create my account?'

In [91]:
nlp = spacy.load('en_core_web_sm')

vector = get_vector(query)
len(vector)

96

In [ ]:
search_query_knn = {
    "knn": {
        "field": "embedding",
        "query_vector": vector,
        "k": 5,
        "num_candidates": 10000,
    },
    "size": 5,
    "_source": ['document_id', 'question', 'answer'],
}

index_name="documents_spacy"
es_results = es_client.search(index=index_name, body=search_query_knn)

[elem['_score'] for elem in es_results["hits"]["hits"]]

[0.7769791, 0.7687049, 0.75467646, 0.7458626, 0.7372478]

In [ ]:
# Vector Search in documents_spacy index.
[hit["_source"] for hit in es_results["hits"]["hits"]]

[{'document_id': 'doc_4_can_i_cancel_my_order_',
  'question': 'Can I cancel my order?',
  'answer': 'You can cancel your order if it has not been shipped yet. Please contact our customer support team with your order details, and we will assist you with the cancellation process.'},
 {'document_id': 'doc_2_how_can_i_track_my_order_',
  'question': 'How can I track my order?',
  'answer': "You can track your order by logging into your account and navigating to the 'Order History' section. There, you will find the tracking information for your shipment."},
 {'document_id': 'doc_8_can_i_change_my_shipping_addre',
  'question': 'Can I change my shipping address after placing an order?',
  'answer': 'If you need to change your shipping address, please contact our customer support team as soon as possible. We will do our best to update the address if the order has not been shipped yet.'},
 {'document_id': 'doc_15_do_you_have_a_loyalty_program_',
  'question': 'Do you have a loyalty program?',

In [98]:
vector_st = model.encode(query)

search_query_knn = {
    "knn": {
        "field": "embedding",
        "query_vector": vector_st,
        "k": 5,
        "num_candidates": 10000,
    },
    "size": 5,
    "_source": ['document_id', 'question', 'answer'],
}

index_name="documents_st"
es_results = es_client.search(index=index_name, body=search_query_knn)

[elem['_score'] for elem in es_results["hits"]["hits"]]

[0.90626216, 0.706826, 0.6207144, 0.56382585, 0.5634005]

In [99]:
# Vector Search in documents_st index.
[hit["_source"] for hit in es_results["hits"]["hits"]]

[{'document_id': 'doc_0_how_can_i_create_an_account_',
  'question': 'How can I create an account?',
  'answer': "To create an account, click on the 'Sign Up' button on the top right corner of our website and follow the instructions to complete the registration process."},
 {'document_id': 'doc_16_can_i_order_without_creating_a',
  'question': 'Can I order without creating an account?',
  'answer': 'Yes, you can place an order as a guest without creating an account. However, creating an account offers benefits such as order tracking and easier future purchases.'},
 {'document_id': 'doc_15_do_you_have_a_loyalty_program_',
  'question': 'Do you have a loyalty program?',
  'answer': 'Yes, we have a loyalty program where you can earn points for every purchase. These points can be redeemed for discounts on future orders. Please visit our website to learn more and join the program.'},
 {'document_id': 'doc_28_what_should_i_do_if_my_discoun',
  'question': 'What should I do if my discount cod

In [100]:
if isinstance(model, SentenceTransformer):
    print("model is an instance of SentenceTransformer")

model is an instance of SentenceTransformer
